# External Event Triggers

megane の viewer を外部ツール（Plotly 等）のイベントから制御するデモです。

- **フレーム選択**: Plotly のクリックでトラジェクトリーのフレームを切り替え
- **原子選択・計測**: Python から原子を選択し距離・角度・二面角を取得
- **イベントコールバック**: megane 側のイベントを購読して他の widget を更新

In [ ]:
import megane

print(f"megane v{megane.__version__}")

## 1. Plotly 連携: クリックでフレーム選択

エネルギー等の時系列プロットのデータ点をクリックすると、
対応するトラジェクトリーフレームが viewer に表示されます。

In [ ]:
import numpy as np

viewer = megane.MolecularViewer()
viewer.load("../tests/fixtures/1crn.pdb", xtc="../tests/fixtures/1crn_vibration.xtc")
print(f"Loaded trajectory: {viewer.total_frames} frames")

In [ ]:
import plotly.graph_objects as go
import ipywidgets as widgets

# ダミーのエネルギーデータを生成
n_frames = viewer.total_frames
frames = np.arange(n_frames)
energy = -500 + 10 * np.sin(frames * 0.1) + np.random.randn(n_frames) * 2

# Plotly FigureWidget を作成
fig = go.FigureWidget(
    data=[go.Scatter(x=frames, y=energy, mode="lines+markers", name="Energy")],
    layout=go.Layout(
        title="Energy vs Frame (click a point to jump)",
        xaxis_title="Frame",
        yaxis_title="Energy (kJ/mol)",
        height=300,
    ),
)

# クリックイベントを megane のフレーム選択に接続
def on_plotly_click(trace, points, state):
    if points.point_inds:
        viewer.frame_index = points.point_inds[0]

fig.data[0].on_click(on_plotly_click)

# 上下に並べて表示
widgets.VBox([fig, viewer])

## 2. プログラムから原子を選択して計測

`selected_atoms` に原子インデックスのリストを設定すると、
viewer 上で原子がハイライトされ、自動的に計測が行われます。

| 原子数 | 計測 |
|-------|------|
| 2 | 距離 (Å) |
| 3 | 角度 (°) |
| 4 | 二面角 (°) |

In [ ]:
v = megane.MolecularViewer()
v.load("../tests/fixtures/1crn.pdb")
v

In [ ]:
# 2原子 → 距離
v.selected_atoms = [0, 1]
print("Distance:", v.measurement)

In [ ]:
# 3原子 → 角度
v.selected_atoms = [0, 1, 2]
print("Angle:", v.measurement)

In [ ]:
# 4原子 → 二面角
v.selected_atoms = [0, 1, 2, 3]
print("Dihedral:", v.measurement)

In [ ]:
# 選択クリア
v.selected_atoms = []

## 3. イベントコールバック

`on_event()` で megane のイベントを購読できます。
デコレータとしてもメソッド呼び出しとしても使えます。

| イベント名 | 発火タイミング | データ |
|-----------|--------------|-------|
| `frame_change` | `frame_index` 変更時 | `{"frame_index": int}` |
| `selection_change` | `selected_atoms` 変更時 | `{"atoms": [int, ...]}` |
| `measurement` | 計測結果更新時 | `{"type", "value", "label", "atoms"}` |

In [ ]:
cb_viewer = megane.MolecularViewer()
cb_viewer.load("../tests/fixtures/1crn.pdb", xtc="../tests/fixtures/1crn_vibration.xtc")

# デコレータでイベント登録
@cb_viewer.on_event("frame_change")
def on_frame(data):
    print(f"Frame changed: {data['frame_index']}")

@cb_viewer.on_event("measurement")
def on_meas(data):
    if data:
        print(f"Measurement: {data['type']} = {data['label']}")

cb_viewer

In [ ]:
# Python からフレームを変更 → frame_change イベントが発火
cb_viewer.frame_index = 10

In [ ]:
# コールバック解除
cb_viewer.off_event("frame_change", on_frame)
cb_viewer.frame_index = 20  # もう print されない

## 4. 双方向連携: フレーム変更で Plotly マーカーを更新

megane のフレーム変更イベントを使って、
Plotly のグラフ上で現在のフレーム位置を示すマーカーを同期させます。

In [ ]:
sync_viewer = megane.MolecularViewer()
sync_viewer.load("../tests/fixtures/1crn.pdb", xtc="../tests/fixtures/1crn_vibration.xtc")

n = sync_viewer.total_frames
x = np.arange(n)
y = -500 + 10 * np.sin(x * 0.1) + np.random.randn(n) * 2

sync_fig = go.FigureWidget(
    data=[
        go.Scatter(x=x, y=y, mode="lines", name="Energy"),
        go.Scatter(x=[0], y=[y[0]], mode="markers", marker=dict(size=12, color="red"), name="Current"),
    ],
    layout=go.Layout(title="Bidirectional sync", height=300,
                     xaxis_title="Frame", yaxis_title="Energy (kJ/mol)"),
)

# Plotly → megane
def on_click(trace, points, state):
    if points.point_inds:
        sync_viewer.frame_index = points.point_inds[0]

sync_fig.data[0].on_click(on_click)

# megane → Plotly (赤マーカーを更新)
@sync_viewer.on_event("frame_change")
def update_marker(data):
    idx = data["frame_index"]
    with sync_fig.batch_update():
        sync_fig.data[1].x = [idx]
        sync_fig.data[1].y = [y[idx]]

widgets.VBox([sync_fig, sync_viewer])

## 5. 二面角のトラジェクトリー解析

各フレームで二面角を計算し、結果を Plotly にプロットする例です。
`frame_index` と `selected_atoms` を組み合わせて使います。

In [ ]:
import time

dih_viewer = megane.MolecularViewer()
dih_viewer.load("../tests/fixtures/1crn.pdb", xtc="../tests/fixtures/1crn_vibration.xtc")

# 二面角を計測する原子を設定
dih_viewer.selected_atoms = [0, 1, 2, 3]

# 各フレームの二面角を収集
dihedrals = []

@dih_viewer.on_event("measurement")
def collect_dihedral(data):
    if data and data["type"] == "dihedral":
        dihedrals.append(data["value"])

for i in range(dih_viewer.total_frames):
    dih_viewer.frame_index = i
    time.sleep(0.01)  # widget の同期待ち

dih_viewer.off_event("measurement", collect_dihedral)

# 結果をプロット
dih_fig = go.FigureWidget(
    data=[go.Scatter(x=list(range(len(dihedrals))), y=dihedrals, mode="lines+markers")],
    layout=go.Layout(
        title="Dihedral angle along trajectory",
        xaxis_title="Frame",
        yaxis_title="Dihedral (°)",
        height=300,
    ),
)

widgets.VBox([dih_fig, dih_viewer])

## 6. 独自 widget との連携

`ipywidgets` のスライダーなど、任意の widget と `on_event` で連携できます。

In [ ]:
slider_viewer = megane.MolecularViewer()
slider_viewer.load("../tests/fixtures/1crn.pdb", xtc="../tests/fixtures/1crn_vibration.xtc")

slider = widgets.IntSlider(
    value=0, min=0, max=slider_viewer.total_frames - 1,
    description="Frame:", continuous_update=True,
)
label = widgets.Label(value="Frame: 0")

# スライダー → megane
def on_slider(change):
    slider_viewer.frame_index = change["new"]

slider.observe(on_slider, names="value")

# megane → ラベル (ブラウザの Timeline 操作時もラベルが更新される)
@slider_viewer.on_event("frame_change")
def on_frame_for_label(data):
    idx = data["frame_index"]
    label.value = f"Frame: {idx}"
    slider.value = idx  # スライダーも同期

widgets.VBox([widgets.HBox([slider, label]), slider_viewer])